In [193]:
from pathlib import Path
from itertools import batched
import re
from collections import Counter

from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import yaml
from Bio import SeqIO, Entrez

In [194]:
with open(snakemake.input[0], "r") as file:
    plsdb_accessions = {line.strip() for line in file.readlines()}

In [ ]:
with open(snakemake.params.secrets, "r") as file:
    secrets = yaml.safe_load(file)

Entrez.email = secrets["email"]
if "api_key" in secrets.keys():
    Entrez.api_key = secrets["api_key"]

In [ ]:
tries = 0
while tries <= 5:
    try:
        with Entrez.esearch(
            db="nuccore",
            term="cfr[gene] AND topology_circular[prop] AND plasmid[title]", 
            idtype="acc",
            retmax=1000,
            maxdate="2024/10/17"
            ) as handle:
            record = Entrez.read(handle)
        break
    except Exception as e:
        print(e)
        tries += 1
        if tries > 5:
            print("tries exceeded")
accessions = set(record["IdList"])
len(accessions)

251

In [197]:
accessions.update(plsdb_accessions)
len(accessions)

337

In [198]:
accessions = {accession.replace("NZ_", "") for accession in accessions}
len(accessions)

240

In [199]:
results = []
failed = []
for accession in accessions:
    tries = 0
    while tries < 3:
        try:
            with Entrez.elink(dbfrom="nuccore", db="biosample", id=accession, idtype="acc") as handle:
                links = Entrez.read(handle, validate=True)
                results.extend([{"nuccore_acc" : link["IdList"][0], "biosample_uid" : link["LinkSetDb"][0]["Link"][0]["Id"]} for link in links])
            break
        except Exception as e:
            tries += 1
            if tries >= 3:
                print("tries exceeded")
                failed.append(accession)

tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded
tries exceeded


In [200]:
accessions = pd.DataFrame(results)

In [201]:
def parse_xml(entries) -> pd.DataFrame:
    df = []
    soup = BeautifulSoup(entries, features="xml")
    for sample in soup("BioSample"):
        metadata = sample.attrs
        ids = {entry.get("db") : entry.text for entry in sample("Id")}
        organism = {"organism" : hit.text if (hit:=sample.find("OrganismName")) else ""}
        attributes = {entry.get("harmonized_name") : entry.text for entry in sample("Attribute")}
        data = {**metadata, **ids, **organism, **attributes}
        df.append(data)
    return pd.DataFrame(df)

In [202]:
with Entrez.efetch(db="biosample", id=accessions["biosample_uid"].unique(), rettype="xml") as handle:
    entries = handle.read()
biosample = parse_xml(entries)

In [203]:
nuccore = []
for accession in failed:
    try:
        with Entrez.efetch(db="nuccore", id=accession, idtype="acc", rettype="xml") as handle:
            entries = handle.read()
        soup = BeautifulSoup(entries, features="xml")
        nuccore.append(pd.DataFrame({qualifier.GBQualifier_name.text : qualifier.GBQualifier_value.text for qualifier in soup.find("GBFeature_quals").find_all("GBQualifier")}, index=[accession]))
    except:
        print(accession)
nuccore = pd.concat(nuccore).reset_index(names="nuccore_acc")

In [205]:
accessions = accessions.merge(biosample, left_on="biosample_uid", right_on="id")
accessions = pd.concat([accessions, nuccore], ignore_index=True)

In [206]:
accessions.columns = [col.lower() if col is not None else "unknown_column_header" for col in accessions.columns]
duplicate_columns = [col for col, count in Counter(accessions.columns).items() if count > 1]
for dupe in duplicate_columns:
    idx = [i for i, col in enumerate(accessions.columns) if col == dupe]
    for i in idx[1:]:
        accessions.iloc[:,idx[0]] = accessions.iloc[:,idx[0]].fillna(accessions.iloc[:,i])
    accessions.drop(accessions.columns[idx[1:]], axis=1, inplace=True)

In [207]:
def merge_columns(column_list: list) -> pd.Series:
        new_column = pd.Series(index=accessions.index)
        for column in column_list:
            if column in accessions.columns:
                new_column = new_column.fillna(accessions[column])
                accessions.drop(column, axis=1, inplace=True)
        return new_column

In [ ]:
species_cols = ["species", "scientific_name", "organism", 'common name']
accessions['organism'] = merge_columns(species_cols)

country_cols = ['geo_loc_name', 'geographic location (country and/or sea)', 'geographic location (region and locality)', 'country', 'geographic location (country and/or sea,region)', 'geographic location']
accessions["country"] = merge_columns(country_cols)

strain_cols = ['strain', 'isolate', 'sample name', 'isolate name alias']
accessions['strain'] = merge_columns(strain_cols)

genotype_cols = ['genotype', 'mlst', 'mlst#', 'mlst# and scheme', 'mlst type', 'sequence type']
accessions['sequence_type'] = merge_columns(genotype_cols)

host_cols = ['host', 'host scientific name', 'host_description', 'nat-host', 'nat_host', 'specific_host', 'isolate host', 'specific host']
accessions['host'] = merge_columns(host_cols)

collection_cols = ['collection_date', 'collection date', 'collection-date']
accessions['collection_date'] = merge_columns(collection_cols)

source_cols = ['isolation_source', 'isolation-source', 'sample_type', 'culture_collection', 'isolation source', 'source_type', 'sample type']
accessions['source'] = merge_columns(source_cols)

accessions[["country", "region"]] = accessions["country"].str.split(":", expand=True, n=1)
accessions["country"] = accessions["country"].str.strip()
accessions["region"] = accessions["region"].str.strip()

missing_data = {column : accessions[column].isna().sum() for column in accessions.columns}
accessions = accessions[[key for key, value in sorted(missing_data.items(), key=lambda item: item[1])]]

In [273]:
accessions = accessions.sort_values("nuccore_acc")
accessions = accessions[~accessions.duplicated(subset="biosample_uid", keep="first") | accessions["biosample_uid"].isnull()]
accessions = accessions[~accessions["strain"].str.contains("RIVM", na=False)]
accessions = accessions.replace("", None)
accessions = accessions[accessions["organism"].notnull()]
accessions = accessions.dropna(axis="columns", thresh=accessions.shape[0] * .75)

In [284]:
taxids = {}
for species in accessions["organism"].unique(): 
    try:
        with Entrez.esearch(term=species, db="taxonomy") as handle:
            results = Entrez.read(handle)
        taxids.update({species : results['IdList'][0]})
    except:
        print(species)

In [292]:
taxonomy = {}
for taxon, taxid in taxids.items():
    with Entrez.efetch(id=taxid, db="taxonomy") as handle:
        results = Entrez.read(handle)
    taxonomy.update({taxon : {rank["Rank"] : rank["ScientificName"] for rank in results[0]['LineageEx']}})
taxonomy = pd.DataFrame(taxonomy).T

In [ ]:
accessions = accessions.merge(taxonomy, left_on="organism", right_index=True, how="left")
accessions = accessions.drop(columns="accession")
accessions = accessions.rename(columns={"nuccore_acc" : "accession"})
accessions.to_csv(snakemake.output.metadata, index=False)

,nuccore_acc,organism,strain,country,biosample_uid,access,publication_date,last_update,submission_date,id,accession,biosample,source,unknown_column_header,cellular root,domain,kingdom,phylum,class,order,family,genus,species,no rank,species group,species subgroup,clade
129,AP026896.1,Aeromonas caviae,KAM479,Japan,31264816,public,2022-10-12T00:00:00.000,2022-10-11T22:03:14.000,2022-10-13T01:01:09.228,31264816,SAMD00535935,SAMD00535935,Microbe,KAM479c,cellular organisms,Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Aeromonadales,Aeromonadaceae,Aeromonas,NaN,NaN,NaN,NaN,NaN
120,CP002208.2,Enterococcus faecalis,EF-01,China,14224439,public,2020-02-27T00:00:00.000,2020-02-27T12:47:29.052,2020-02-27T12:47:29.052,14224439,SAMN14224439,SAMN14224439,bovine,NaN,cellular organisms,Bacteria,Bacillati,Bacillota,Bacilli,Lactobacillales,Enterococcaceae,Enterococcus,NaN,NaN,NaN,NaN,NaN
97,CP012031.1,Leptospira borgpetersenii serovar Ballum,56604,China: West region,3785545,public,2015-11-19T00:00:00.000,2021-02-24T10:06:45.808,2015-06-22T22:36:43.000,3785545,SAMN03785545,SAMN03785545,animal,Leptospira borgpetersenii serogroup Ballum se...,cellular organisms,Bacteria,Pseudomonadati,Spirochaetota,Leptospiria,Leptospirales,Leptospiraceae,Leptospira,Leptospira borgpetersenii,NaN,NaN,NaN,NaN
127,CP017783.1,Rhodobacter xanthinilyticus,LPB0142,South Korea,5726021,public,2016-09-02T00:00:00.000,2024-02-02T21:07:51.035,2016-09-02T01:46:01.930,5726021,SAMN05726021,SAMN05726021,sea water,type strain of Rhodobacter xanthinilyticus,cellular organisms,Bacteria,Pseudomonadati,Pseudomonadota,Alphaproteobacteria,Rhodobacterales,NaN,Rhodobacter,NaN,Rhodobacter group,NaN,NaN,NaN
77,CP025986.1,Ralstonia solanacearum,RSCM,China: Guangdong,8180272,public,2018-01-23T00:00:00.000,2021-02-28T05:13:14.238,2017-12-14T20:24:02.550,8180272,SAMN08180272,SAMN08180272,culture,Ralstonia solanacearum,cellular organisms,Bacteria,Pseudomonadati,Pseudomonadota,Betaproteobacteria,Burkholderiales,Burkholderiaceae,Ralstonia,NaN,NaN,Ralstonia solanacearum species complex,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23,OY725736.1,Monoglobus pectinilyticus,Ruminococcaceae species TSDC17.2-1.2,USA,37507651,public,2023-09-21T00:00:00.000,2023-09-21T16:06:48.000,2023-09-22T08:24:10.653,37507651,SAMEA114266243,SAMEA114266243,NaN,Monoglobus pectinilyticus,cellular organisms,Bacteria,Bacillati,Bacillota,Clostridia,Monoglobales,Monoglobaceae,Monoglobus,NaN,NaN,NaN,NaN,NaN
31,OY725759.1,Parabacteroides distasonis,Parabacteroides distasonis TSDC17.2-1.1,USA,37507650,public,2023-09-21T00:00:00.000,2023-09-21T16:06:45.000,2023-09-22T08:24:10.633,37507650,SAMEA114266242,SAMEA114266242,NaN,Parabacteroides distasonis,cellular organisms,Bacteria,Pseudomonadati,Bacteroidota,Bacteroidia,Bacteroidales,Tannerellaceae,Parabacteroides,NaN,NaN,NaN,NaN,Bacteroidota/Chlorobiota group
26,OY725841.1,Subdoligranulum variabile,Subdoligranulum variabile TSDC17.2-1.1,USA,37507652,public,2023-09-21T00:00:00.000,2023-09-21T16:06:51.000,2023-09-22T08:24:10.670,37507652,SAMEA114266244,SAMEA114266244,NaN,Subdoligranulum variabile,cellular organisms,Bacteria,Bacillati,Bacillota,Clostridia,Eubacteriales,Oscillospiraceae,Subdoligranulum,NaN,NaN,NaN,NaN,NaN
9,OY725907.1,Bacteroides finegoldii,Bacteroides finegoldii TSDC17.2-1.1,USA,37507645,public,2023-09-21T00:00:00.000,2023-09-21T16:06:33.000,2023-09-22T08:24:10.540,37507645,SAMEA114266237,SAMEA114266237,NaN,Bacteroides finegoldii,cellular organisms,Bacteria,Pseudomonadati,Bacteroidota,Bacteroidia,Bacteroidales,Bacteroidaceae,Bacteroides,NaN,NaN,NaN,NaN,Bacteroidota/Chlorobiota group


In [ ]:
directory = Path(snakemake.output.fasta) 
if not directory.exists():
    directory.mkdir()
for accession in accessions["accession"]:
    tries = 0
    while tries < 3:
        try:
            with Entrez.efetch(db="nuccore", id=accession, idtype="acc", rettype="fasta", retmode="text") as handle:
                record = SeqIO.read(handle, "fasta")
                with open(directory / f"{accession}.fasta", "w") as handle:
                    SeqIO.write(record, handle, "fasta")
            break
        except Exception as e:
            print(e)
            tries += 1
            if tries >=3:
                print(f"tries exceeded, {accession}")